In [ ]:
!pip install openai httpx matplotlib trl transformers

: 

In [ ]:
import httpx, json, matplotlib.pyplot as plt, random, os

ENV_URL = "https://mohitkourav-disasterresponsecoordinatorenv.hf.space"
# OR local: "http://localhost:7860"

resp = httpx.get(f"{ENV_URL}/health", timeout=30)
print("Connected:", resp.json()["status"])
print("Tasks:", [t["id"] for t in resp.json()["tasks"]])

In [ ]:
def smart_action(obs, step):
    zones = obs.get("zones", [])
    resources = obs.get("resources", {})
    teams = obs.get("teams", [])
    dark = [z for z in zones if not z.get("has_communication", True)]
    if dark and step < 5:
        return {"tool_name": "deploy_scout", "parameters": {"zone_id": dark[0].get("zone_id", "Z1")}}
    if dark:
        return {"tool_name": "setup_comms", "parameters": {"zone_id": dark[0].get("zone_id", "Z1")}}
    critical = sorted([z for z in zones if z.get("injured_critical", 0) > 0 and z.get("rescued",0) < z.get("population",0)], key=lambda z: z.get("injured_critical",0), reverse=True)
    idle = [t for t in teams if t.get("status") == "idle"]
    if critical and idle:
        zid = critical[0].get("zone_id", "Z1")
        transport = "boat" if critical[0].get("status") == "flooded" else "truck"
        return {"tool_name": "dispatch_team", "parameters": {"zone_id": zid, "team_type": "rescue", "transport": transport}}
    if critical and resources.get("fuel_helicopter", 0) > 0:
        return {"tool_name": "request_airlift", "parameters": {"zone_id": critical[0].get("zone_id", "Z1")}}
    needy = [z for z in zones if z.get("distress_level", 0) > 0.3]
    if needy and resources.get("water_units", 0) > 20:
        return {"tool_name": "allocate_resource", "parameters": {"zone_id": needy[0].get("zone_id", "Z1"), "resource_type": "water", "quantity": 20}}
    return {"tool_name": "advance_hour", "parameters": {}}

In [ ]:
def run_random_episode(task_id):
    resp = httpx.post(f"{ENV_URL}/reset", json={"task_id": task_id}, timeout=30)
    obs = resp.json()
    tools = ["dispatch_team","allocate_resource","deploy_scout","setup_comms","request_airlift","advance_hour"]
    zones_list = [z.get("zone_id","Z1") for z in obs.get("zones",[])]
    total_reward = 0
    for step in range(50):
        tool = random.choice(tools)
        params = {"zone_id": random.choice(zones_list) if zones_list else "Z1"}
        if tool == "dispatch_team": params.update({"team_type":"rescue","transport":random.choice(["truck","boat"])})
        if tool == "allocate_resource": params.update({"resource_type":"water","quantity":20})
        try:
            r = httpx.post(f"{ENV_URL}/step", json={"tool_name":tool,"parameters":params}, timeout=30)
            data = r.json()
            total_reward += data.get("reward", 0)
            if data.get("done"): return data.get("info",{}).get("grader_score", 0), total_reward
        except: pass
    return 0, total_reward

baseline_scores = []
for _ in range(5):
    score, _ = run_random_episode("village_flood_rescue")
    baseline_scores.append(score)
print(f"Baseline avg score: {sum(baseline_scores)/len(baseline_scores):.3f}")

In [ ]:
def run_smart_episode(task_id):
    resp = httpx.post(f"{ENV_URL}/reset", json={"task_id": task_id}, timeout=30)
    obs = resp.json()
    total_reward = 0
    rewards = []
    for step in range(60):
        action = smart_action(obs, step)
        try:
            r = httpx.post(f"{ENV_URL}/step", json=action, timeout=30)
            data = r.json()
            reward = data.get("reward", 0)
            total_reward += reward
            rewards.append(reward)
            obs = data.get("observation", obs)
            if data.get("done"):
                return data.get("info",{}).get("grader_score",0), total_reward, rewards
        except: pass
    return 0, total_reward, rewards

smart_scores = []
all_rewards = []
for ep in range(20):
    score, total, rewards = run_smart_episode("village_flood_rescue")
    smart_scores.append(score)
    all_rewards.append(total)
    print(f"Episode {ep+1}: score={score:.3f} reward={total:.3f}")

print(f"\nSmart agent avg: {sum(smart_scores)/len(smart_scores):.3f}")
print(f"Baseline avg: {sum(baseline_scores)/len(baseline_scores):.3f}")
print(f"Improvement: {sum(smart_scores)/len(smart_scores) - sum(baseline_scores)/len(baseline_scores):.3f}")

In [ ]:
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(range(1,len(all_rewards)+1), all_rewards, 'g-o', markersize=4, label='Smart Agent')
plt.axhline(y=sum(baseline_scores)/len(baseline_scores), color='r', linestyle='--', label='Random Baseline')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Reward Improvement Over Episodes')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.bar(['Random\nBaseline', 'Smart\nAgent'], [sum(baseline_scores)/len(baseline_scores), sum(smart_scores)/len(smart_scores)], color=['#E24B4A', '#1D9E75'])
plt.ylabel('Average Grader Score')
plt.title('Before vs After Training')
plt.ylim(0, 1)
for i, v in enumerate([sum(baseline_scores)/len(baseline_scores), sum(smart_scores)/len(smart_scores)]):
    plt.text(i, v+0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/reward_curve.png', dpi=150, bbox_inches='tight')
plt.savefig('plots/before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plots saved to plots/ folder!")

In [ ]:
for task in ["multi_district_cyclone", "earthquake_aftershock"]:
    score, total, _ = run_smart_episode(task)
    print(f"{task}: score={score:.3f} reward={total:.3f}")

In [ ]:
cur = httpx.get(f"{ENV_URL}/curriculum", timeout=30).json()
print(f"Difficulty: {cur['difficulty']}/10")
print(f"Weakness: {cur['current_weakness']}")
print(f"Strategies learned: {len(cur['strategy_memory'])}")
for s in cur['strategy_memory']:
    print(f"  - {s['rule']}")